# 实验分析 - 2022世界杯版本

本notebook提供完整的实验分析功能，用于探索性分析2022世界杯比赛中的防守表现数据。

**适配版本**: 2022 FIFA世界杯数据集

## 功能概述

1. **数据加载与合并** - 加载多场比赛的球员评估结果
2. **网格/区域分析** - 分析不同球场区域的防守表现
3. **防守组合分析** - 评估防守球员配对的效果
4. **球员排名** - 综合评估和排名防守球员
5. **统计分析** - 相关性分析和描述统计
6. **可视化分析** - 生成各类分析图表

## 1. 导入库和配置

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from mplsoccer import Pitch
import convert_tracking as ct
from tqdm import tqdm

# 配置matplotlib中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

print("✓ 库导入成功")
print("✓ 中文字体配置完成")

## 2. 配置参数

In [ ]:
# 配置要分析的比赛ID列表
# 方式1: 分析单场比赛
GAME_IDS = ['10517']  # 决赛

# 方式2: 分析多场比赛
# GAME_IDS = ['10517', '3857', '3859']

# 方式3: 分析所有可用比赛
# GAME_IDS = ct.get_available_games()

XT_GRID_PATH = 'xT_grid.csv'
OUTPUT_DIR = 'results/experiments_2022WC'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"配置完成: {len(GAME_IDS)} 场比赛")
print(f"比赛ID: {GAME_IDS}")
print(f"输出目录: {OUTPUT_DIR}")

## 3. 创建球员位置映射

In [ ]:
# 从所有比赛中收集球员位置信息
player_position_dict = {}

for game_id in GAME_IDS:
    data_dir = f'Data/{game_id}'
    players_file = f'{data_dir}/players_{game_id}.csv'
    
    if os.path.exists(players_file):
        players_df = pd.read_csv(players_file)
        for _, row in players_df[['playerName', 'playerPos']].drop_duplicates().iterrows():
            player_position_dict[row['playerName']] = row['playerPos']

print(f"✓ 创建了 {len(player_position_dict)} 个球员的位置映射")
print(f"位置类型: {set(player_position_dict.values())}")

## 4. 加载球员评估结果

In [ ]:
# 加载所有比赛的球员评估结果
results_model1 = []
match_id_error = []

for game_id in tqdm(GAME_IDS, desc="加载评估结果"):
    results_file = f'results/player_eval_{game_id}/{game_id}_defender_performance.csv'
    
    try:
        if os.path.exists(results_file):
            df = pd.read_csv(results_file)
            results_model1.append(df)
        else:
            print(f"警告: 未找到 {game_id} 的结果文件")
            match_id_error.append(game_id)
    except Exception as e:
        print(f"错误: 加载 {game_id} 失败 - {e}")
        match_id_error.append(game_id)

if results_model1:
    results_model1_df = pd.concat(results_model1, ignore_index=True)
    print(f"\n✓ 成功加载 {len(results_model1)} 场比赛的数据")
    print(f"总记录数: {len(results_model1_df):,}")
    print(f"唯一防守球员: {results_model1_df['Defender'].nunique()}")
    print(f"唯一进攻球员: {results_model1_df['Attacker'].nunique()}")
    
    if match_id_error:
        print(f"\n失败的比赛ID: {match_id_error}")
else:
    print("❌ 未能加载任何数据")
    print("请先运行 PlayerEval_2022WC.ipynb 生成评估结果")

## 5. 数据预处理

In [ ]:
if 'results_model1_df' in locals() and results_model1_df is not None:
    # 添加球员位置信息
    results_model1_df['defender_pos'] = results_model1_df['Defender'].map(player_position_dict)
    results_model1_df['attacker_pos'] = results_model1_df['Attacker'].map(player_position_dict)
    
    # 计算相对表现
    avg_performance = results_model1_df.groupby(['match_id', 'graph_index'])['Total_Perf'].transform('mean')
    results_model1_df['relative_performance'] = results_model1_df['Total_Perf'] - avg_performance
    
    # 创建唯一记录
    results_model1_df_unique = results_model1_df.drop_duplicates(subset=['match_id', 'graph_index', 'Defender'])
    
    print("✓ 数据预处理完成")
    print(f"唯一记录数: {len(results_model1_df_unique):,}")
    print(f"\n防守球员位置分布:")
    print(results_model1_df['defender_pos'].value_counts())

## 6. 保存合并后的数据

In [ ]:
if 'results_model1_df' in locals() and results_model1_df is not None:
    results_model1_df.to_csv(f'{OUTPUT_DIR}/results_model_df.csv', index=False)
    results_model1_df_unique.to_csv(f'{OUTPUT_DIR}/results_model_df_unique.csv', index=False)
    print(f"✓ 数据已保存到 {OUTPUT_DIR}")

## 7. 防守球员表现排名

In [ ]:
if 'results_model1_df_unique' in locals():
    grouped_defender_values = results_model1_df_unique.groupby('Defender').agg({
        'defender_pos': 'first',
        'Total_Perf': ['sum', 'mean', 'std', 'count']
    }).round(4)
    
    grouped_defender_values.columns = ['_'.join(str(c) for c in col).strip('_') for col in grouped_defender_values.columns.values]
    grouped_defender_values = grouped_defender_values.sort_values('Total_Perf_mean', ascending=False)
    
    print("防守球员表现排名 (Top 20):")
    print(grouped_defender_values.head(20))
    
    grouped_defender_values.to_csv(f'{OUTPUT_DIR}/defender_rankings.csv')
    print(f"\n✓ 排名已保存")

## 8. 按位置分析

In [ ]:
if 'results_model1_df_unique' in locals():
    def sem(x):
        return 1.96 * (x.std() / np.sqrt(len(x)))
    
    grouped_position_values = results_model1_df_unique.groupby('defender_pos').agg({
        'Total_Perf': ['mean', 'std', 'count', sem]
    }).round(4)
    
    print("按位置统计:")
    print(grouped_position_values)
    
    # 可视化
    fig, ax = plt.subplots(figsize=(10, 6))
    positions = grouped_position_values.index
    means = grouped_position_values[('Total_Perf', 'mean')]
    sems = grouped_position_values[('Total_Perf', 'sem')]
    
    ax.bar(positions, means, yerr=sems, capsize=5, alpha=0.7)
    ax.set_xlabel('位置', fontsize=12)
    ax.set_ylabel('平均防守表现', fontsize=12)
    ax.set_title('不同位置的防守表现', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/position_performance.png', dpi=150)
    plt.show()

## 9. 防守组合分析

In [ ]:
if 'results_model1_df_unique' in locals():
    cb_data = results_model1_df_unique[results_model1_df_unique['defender_pos'] == 'D']
    
    if len(cb_data) > 0:
        cb_pairs = []
        for (match_id, graph_idx), group in cb_data.groupby(['match_id', 'graph_index']):
            if len(group) >= 2:
                defenders = sorted(group['Defender'].tolist())
                pair_name = ' + '.join(defenders[:2])
                total_perf = group['Total_Perf'].iloc[:2].sum()
                cb_pairs.append({
                    'CB_Pair': pair_name,
                    'Summed_Total_Perf': total_perf
                })
        
        if cb_pairs:
            cb_duo_df = pd.DataFrame(cb_pairs)
            cb_duo_ranking = cb_duo_df.groupby('CB_Pair').agg({
                'Summed_Total_Perf': ['sum', 'mean', 'count']
            }).round(4)
            cb_duo_ranking = cb_duo_ranking.sort_values(('Summed_Total_Perf', 'sum'), ascending=False)
            
            print("中后卫组合排名 (Top 10):")
            print(cb_duo_ranking.head(10))
            
            cb_duo_ranking.to_csv(f'{OUTPUT_DIR}/cb_duo_rankings.csv')
            print(f"\n✓ 已保存")

## 10. 相关性分析

In [ ]:
if 'results_model1_df' in locals():
    correlation_cols = ['Def_Distance', 'Def_Attention', 'Def_Influence', 'Att_Threat', 'Total_Perf']
    available_cols = [col for col in correlation_cols if col in results_model1_df.columns]
    
    if len(available_cols) > 1:
        corr_matrix = results_model1_df[available_cols].corr()
        
        print("相关性矩阵:")
        print(corr_matrix.round(3))
        
        # 可视化
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
        
        for i in range(len(available_cols)):
            for j in range(len(available_cols)):
                ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black")
        
        ax.set_xticks(range(len(available_cols)))
        ax.set_yticks(range(len(available_cols)))
        ax.set_xticklabels(available_cols, rotation=45, ha='right')
        ax.set_yticklabels(available_cols)
        ax.set_title('指标相关性矩阵', fontsize=14)
        
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/correlation_matrix.png', dpi=150)
        plt.show()

## 11. 散点图分析

In [ ]:
if 'results_model1_df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].scatter(results_model1_df['Def_Distance'], 
                   results_model1_df['Def_Influence'], 
                   alpha=0.2, s=5)
    axes[0].set_xlabel('防守距离 (m)')
    axes[0].set_ylabel('防守影响力')
    axes[0].set_title('距离 vs 影响力')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].scatter(results_model1_df['Def_Attention'], 
                   results_model1_df['Def_Influence'], 
                   alpha=0.2, s=5)
    axes[1].set_xlabel('注意力权重')
    axes[1].set_ylabel('防守影响力')
    axes[1].set_title('注意力 vs 影响力')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/scatter_analysis.png', dpi=150)
    plt.show()
    
    corr_dist = results_model1_df[['Def_Distance', 'Def_Influence']].corr().iloc[0, 1]
    corr_attn = results_model1_df[['Def_Attention', 'Def_Influence']].corr().iloc[0, 1]
    
    print(f"距离与影响力相关系数: {corr_dist:.4f}")
    print(f"注意力与影响力相关系数: {corr_attn:.4f}")

## 总结

本notebook完成了以下分析：

✓ 数据加载和合并
✓ 球员表现排名
✓ 位置分析
✓ 防守组合分析
✓ 相关性分析
✓ 可视化分析

### 输出文件：

- `results_model_df.csv` - 完整数据
- `results_model_df_unique.csv` - 唯一记录
- `defender_rankings.csv` - 球员排名
- `cb_duo_rankings.csv` - 中后卫组合排名
- 各类可视化图表

### 下一步：

- 使用 `test_experiments_final.ipynb` 进行快速测试
- 分析更多比赛数据
- 进行更深入的统计建模

### 相关文件：

- `test_experiments_final.ipynb` - 测试版本
- `PlayerEval_2022WC.ipynb` - 球员评估工具
- `Visualisation_2022WC.ipynb` - 可视化工具